In [4]:
# Dashboard developed by Allen Eichhorn for CS 340 Project Two.

from pathlib import Path
import base64
import re

import pandas as pd
import plotly.express as px
import dash_leaflet as dl

from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()


###########################
# Data Manipulation / Model
###########################

# Use the same account and database settings as the milestone.
shelter = AnimalShelter(
    username="aacuser",
    password="YOUR_MONGODB_PASSWORD",
    host="localhost",
    port=27017,
    database_name="aac",
    collection_name="animals",
    auth_source="admin"
)

# Keep this connection available because filters query MongoDB.
initial_records = shelter.read({})

if not initial_records:
    raise RuntimeError(
        "No shelter records were retrieved. Check MongoDB, "
        "your credentials, and the aac.animals collection."
    )


def prepare_records(records):
    """Remove MongoDB IDs and make missing values safe for Dash."""
    frame = pd.DataFrame.from_records(records)
    frame.drop(columns=["_id"], errors="ignore", inplace=True)
    frame = frame.astype(object).where(pd.notnull(frame), None)
    return frame.to_dict("records")


initial_data = prepare_records(initial_records)
table_columns = list(pd.DataFrame(initial_data).columns)

# Required breeds, sexes, and age ranges from the specifications.
rescue_profiles = {
    "water": {
        "breeds": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ],
        "sex": "Intact Female",
        "min_age": 26,
        "max_age": 156
    },
    "mountain": {
        "breeds": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "min_age": 26,
        "max_age": 156
    },
    "disaster": {
        "breeds": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "min_age": 20,
        "max_age": 300
    }
}


def build_query(filter_type):
    """Build a MongoDB query for the selected rescue category."""
    if filter_type == "reset":
        return {}

    profile = rescue_profiles[filter_type]

    # Match preferred breeds, including mix and crossbreed records.
    breed_pattern = "|".join(
        re.escape(breed) for breed in profile["breeds"]
    )

    return {
        "animal_type": "Dog",
        "breed": {
            "$regex": breed_pattern,
            "$options": "i"
        },
        "sex_upon_outcome": profile["sex"],
        "age_upon_outcome_in_weeks": {
            "$gte": profile["min_age"],
            "$lte": profile["max_age"]
        }
    }


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Look beside the notebook first, then in common Codio folders.
logo_name = "Grazioso Salvare Logo.png"
logo_candidates = [
    Path(logo_name),
    Path("code_files") / logo_name,
    Path("../code_files") / logo_name,
    Path("/home/codio/workspace/code_files") / logo_name
]

logo_path = next(
    (path for path in logo_candidates if path.is_file()),
    None
)

if logo_path is None:
    raise FileNotFoundError(
        "Place Grazioso Salvare Logo.png in the same folder "
        "as this notebook, then run it again."
    )

encoded_logo = base64.b64encode(
    logo_path.read_bytes()
).decode("utf-8")

app.layout = html.Div(
    [
        html.Div(
            [
                html.A(
                    html.Img(
                        src=f"data:image/png;base64,{encoded_logo}",
                        alt="Grazioso Salvare logo",
                        style={
                            "width": "170px",
                            "height": "100px",
                            "objectFit": "contain"
                        }
                    ),
                    href="https://www.snhu.edu",
                    target="_blank"
                ),
                html.Div(
                    [
                        html.H1(
                            "Grazioso Salvare Dashboard",
                            style={"margin": "0"}
                        ),
                        html.P(
                            "Developed by Allen Eichhorn",
                            style={"marginBottom": "0"}
                        )
                    ]
                )
            ],
            style={
                "display": "flex",
                "alignItems": "center",
                "gap": "24px"
            }
        ),

        html.Hr(),

        html.H3("Find Search-and-Rescue Candidates"),

        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Reset / All Animals", "value": "reset"},
                {"label": "Water Rescue", "value": "water"},
                {
                    "label": "Mountain or Wilderness Rescue",
                    "value": "mountain"
                },
                {
                    "label": "Disaster or Individual Tracking",
                    "value": "disaster"
                }
            ],
            value="reset",
            labelStyle={
                "display": "inline-block",
                "marginRight": "24px",
                "marginBottom": "8px"
            },
            inputStyle={"marginRight": "6px"}
        ),

        html.P(
            id="record-count",
            children=f"Showing {len(initial_data):,} animal records."
        ),

        html.P(
            "Select a row to view the animal's location. "
            "Click column headings to sort the records."
        ),

        dash_table.DataTable(
            id="datatable-id",
            columns=[
                {
                    "name": column.replace("_", " ").title(),
                    "id": column,
                    "selectable": True
                }
                for column in table_columns
            ],
            data=initial_data,
            row_selectable="single",
            selected_rows=[0],
            column_selectable="single",
            selected_columns=[],
            page_action="native",
            page_current=0,
            page_size=5,
            sort_action="native",
            sort_mode="multi",
            sort_by=[],
            style_table={"overflowX": "auto"},
            style_cell={
                "fontFamily": "Arial",
                "fontSize": "12px",
                "textAlign": "left",
                "padding": "6px",
                "minWidth": "100px",
                "maxWidth": "220px",
                "whiteSpace": "normal"
            },
            style_header={
                "backgroundColor": "#e8eef4",
                "fontWeight": "bold"
            }
        ),

        html.Hr(),

        html.Div(
            [
                html.Div(
                    [
                        html.H3("Breed Distribution"),
                        html.Div(id="graph-id")
                    ],
                    style={
                        "flex": "1 1 420px",
                        "minWidth": "0"
                    }
                ),
                html.Div(
                    [
                        html.H3("Selected Animal Location"),
                        html.Div(id="map-id")
                    ],
                    style={
                        "flex": "1 1 420px",
                        "minWidth": "0"
                    }
                )
            ],
            style={
                "display": "flex",
                "flexWrap": "wrap",
                "gap": "24px"
            }
        )
    ],
    style={
        "fontFamily": "Arial",
        "padding": "16px",
        "maxWidth": "1500px",
        "margin": "auto"
    }
)


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "selected_rows"),
        Output("datatable-id", "page_current"),
        Output("datatable-id", "sort_by"),
        Output("datatable-id", "selected_columns"),
        Output("record-count", "children")
    ],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """Query MongoDB and reset table controls when filters change."""
    records = shelter.read(build_query(filter_type))
    data = prepare_records(records)

    selected_rows = [0] if data else []

    return (
        data,
        selected_rows,
        0,
        [],
        [],
        f"Showing {len(data):,} animal records."
    )


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(view_data):
    """Summarize all current table records, across all pages."""
    if view_data is None:
        view_data = initial_data

    if not view_data:
        return html.P("No animals match this rescue profile.")

    frame = pd.DataFrame.from_records(view_data)
    breeds = frame["breed"].fillna("Unknown breed")
    counts = breeds.value_counts()

    # Keep the unfiltered chart readable by grouping smaller categories.
    if len(counts) > 10:
        chart_data = pd.DataFrame({
            "Breed": list(counts.index[:10]) + ["Other breeds"],
            "Animals": list(counts.iloc[:10]) + [
                int(counts.iloc[10:].sum())
            ]
        })
    else:
        chart_data = pd.DataFrame({
            "Breed": counts.index,
            "Animals": counts.values
        })

    figure = px.pie(
        chart_data,
        names="Breed",
        values="Animals",
        hole=0.35
    )

    figure.update_traces(textinfo="percent")
    figure.update_layout(
        height=350,
        margin={"l": 10, "r": 10, "t": 10, "b": 10},
        legend={"font": {"size": 10}}
    )

    return dcc.Graph(
        figure=figure,
        config={"displayModeBar": False}
    )


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    """Highlight the selected column and selected animal."""
    styles = [
        {
            "if": {"column_id": column},
            "backgroundColor": "#D2F3FF"
        }
        for column in (selected_columns or [])
    ]

    styles.append({
        "if": {"state": "selected"},
        "backgroundColor": "#D2F3FF",
        "border": "1px solid #3896c4"
    })

    return styles


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(view_data, selected_rows):
    """Display the selected animal using named coordinate fields."""
    if view_data is None:
        view_data = initial_data

    if not view_data:
        return html.P("No animal locations are available.")

    row_index = selected_rows[0] if selected_rows else 0

    if not 0 <= row_index < len(view_data):
        row_index = 0

    animal = view_data[row_index]

    try:
        latitude = float(animal["location_lat"])
        longitude = float(animal["location_long"])
    except (KeyError, TypeError, ValueError):
        return html.P(
            "The selected animal has no valid location coordinates."
        )

    if not (
        -90 <= latitude <= 90
        and -180 <= longitude <= 180
    ):
        return html.P(
            "The selected animal has no valid location coordinates."
        )

    name = str(animal.get("name") or "Unnamed animal")
    breed = str(animal.get("breed") or "Unknown breed")
    animal_id = str(animal.get("animal_id") or "Unknown ID")

    return dl.Map(
        center=[latitude, longitude],
        zoom=10,
        style={"width": "100%", "height": "350px"},
        children=[
            dl.TileLayer(),
            dl.Marker(
                position=[latitude, longitude],
                children=[
                    dl.Tooltip(f"{name} — {breed}"),
                    dl.Popup([
                        html.H4(name),
                        html.P(f"Breed: {breed}"),
                        html.P(f"Animal ID: {animal_id}")
                    ])
                ]
            )
        ]
    )


# If port 8050 is already being used, change it to 8051.
app.run_server(
    mode="inline",
    port=8050,
    debug=False,
    height=1100
)

 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [19/Oct/2026 05:54:22] "GET /_alive_b0e0e4f7-aae9-4980-a3b2-40ae7749c46a HTTP/1.1" 200 -


127.0.0.1 - - [19/Oct/2026 05:54:22] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 05:54:23] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 05:54:23] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 05:54:23] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Oct/2026 05:54:23] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 05:54:23] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Oct/2026 05:54:23] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 05:54:24] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 05:54:25] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 05:54:25] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 05:54:25] "GET /_dash-component-suites/dash/dcc/async-plotlyjs.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Oct/2026 05:54:25]